# Решения. Отчёт по таблице объявлений

Эталон итоговой пары. Ученикам до сдачи не показывать.


In [ ]:
from pathlib import Path
import pandas as pd

LISTINGS_URL = (
    "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/"
    "modules/08_02_carsharing_pandas_lr/data/listings.csv"
)

def find_listings_csv():
    for p in (
        Path("listings.csv"),
        Path("../listings.csv"),
        Path("../../data/listings.csv"),
        Path("../../../data/listings.csv"),
        Path("../data/listings.csv"),
    ):
        if p.exists():
            return p.resolve()
    return LISTINGS_URL

LISTINGS_PATH = find_listings_csv()
df = pd.read_csv(LISTINGS_PATH)


## 1. На чём держится отчёт


In [ ]:
n_rows = int(len(df))
n_cols = int(df.shape[1])
numeric = ['accommodates', 'bedrooms', 'number_of_reviews', 'price']
REPORT_TABLE = (
    f'Отчёт построен на {n_rows} объявлениях и {n_cols} столбцах. '
    'Объявление описывают listing_id, neighbourhood, room_type, accommodates, bedrooms '
    'и number_of_reviews, а цену ночи дальше разбираем по столбцу price.'
)
print(n_rows, n_cols)
print(REPORT_TABLE)


## 2. Пропуски


In [ ]:
missing = df.isna().sum()
gaps = [f'{name}: {int(n)}' for name, n in missing.items() if int(n) > 0]
if gaps:
    REPORT_NA = 'Пропуски есть в столбцах ' + ', '.join(gaps) + '.'
else:
    REPORT_NA = 'Пустых ячеек нет ни в одном столбце.'
print(missing)
print(REPORT_NA)


## 3. Цена по всей таблице


In [ ]:
price_mean = float(df['price'].mean())
price_median = float(df['price'].median())
price_min = float(df['price'].min())
price_max = float(df['price'].max())
REPORT_PRICE = (
    f'Средняя цена {price_mean:.1f}, медиана {price_median:.1f}, '
    f'минимум {price_min:.0f}, максимум {price_max:.0f}. '
    f'Среднее выше медианы, потому что дорогой хвост тянет его вверх. '
    f'Типичной ценой ночи беру медиану {price_median:.0f}: половина объявлений не дороже неё.'
)
print(REPORT_PRICE)


## 4. Объявления хотя бы на четырёх гостей


In [ ]:
fits = df.loc[df['accommodates'] >= 4]
n_fits_four = int(len(fits))
median_fits_four = float(fits['price'].median())
first_area_fits = str(fits['neighbourhood'].iloc[0])
REPORT_SLICE = (
    f'Объявлений хотя бы на четырёх гостей {n_fits_four}, '
    f'медиана их цены {median_fits_four:.1f}, '
    f'район первой строки среза — {first_area_fits}.'
)
print(REPORT_SLICE)


## 5. Каких типов комнат больше


In [ ]:
room_counts = df['room_type'].value_counts()
top_room = str(room_counts.idxmax())
n_top_room = int(room_counts.max())
REPORT_ROOMS = f'Чаще всего встречается тип {top_room}: таких объявлений {n_top_room}.'
print(room_counts)
print(REPORT_ROOMS)


## 6. Медиана цены у каждого типа комнаты


In [ ]:
median_by_room = df.groupby('room_type')['price'].median()
dearest_room = str(median_by_room.idxmax())
dearest_median = float(median_by_room.max())
REPORT_TYPE = f'Выше всех медиана цены у типа {dearest_room}: {dearest_median:.1f}.'
print(median_by_room)
print(REPORT_TYPE)


## 7. Пять самых дорогих объявлений


In [ ]:
top5 = (
    df.sort_values('price', ascending=False)
    .head(5)[['listing_id', 'neighbourhood', 'room_type', 'price']]
)
row0 = top5.iloc[0]
REPORT_TOP = (
    f'Самое дорогое объявление — {row0.listing_id} в районе {row0.neighbourhood}, '
    f'цена {float(row0.price):.0f}.'
)
print(top5)
print(REPORT_TOP)


## 8. Середина цен


In [ ]:
q1_price = float(df['price'].quantile(0.25))
q3_price = float(df['price'].quantile(0.75))
mid_width = q3_price - q1_price
REPORT_MIDDLE = (
    f'Нижний квартиль цены {q1_price:.1f}, верхний {q3_price:.1f}, '
    f'середина занимает {mid_width:.1f}.'
)
print(REPORT_MIDDLE)


## 9. Какой числовой признак ближе к цене


In [ ]:
corr_acc = float(df['accommodates'].corr(df['price']))
corr_bed = float(df['bedrooms'].corr(df['price']))
closer_feature = 'accommodates' if abs(corr_acc) >= abs(corr_bed) else 'bedrooms'
REPORT_CORR = (
    f'Корреляция числа гостей с ценой {corr_acc:.3f}, числа спален с ценой {corr_bed:.3f}. '
    f'Ближе к цене столбец {closer_feature}. '
    'Эти два числа не доказывают, что гости или спальни создают цену: '
    'они только показывают, какие столбцы меняются вместе.'
)
print(REPORT_CORR)


## 10. Самый наполненный район


In [ ]:
area_counts = df['neighbourhood'].value_counts()
busy_area = str(area_counts.idxmax())
busy = df.loc[df['neighbourhood'] == busy_area]
n_busy = int(len(busy))
median_busy = float(busy['price'].median())
n_busy_above = int((busy['price'] > price_median).sum())
if median_busy > price_median:
    vs = 'выше медианы всей таблицы'
elif median_busy < price_median:
    vs = 'ниже медианы всей таблицы'
else:
    vs = 'совпадает с медианой всей таблицы'
REPORT_AREA = (
    f'Больше всего объявлений в районе {busy_area}: их {n_busy}, '
    f'медиана цены {median_busy:.1f}, дороже медианы всей таблицы {n_busy_above} объявлений. '
    f'Медиана этого района {vs}.'
)
print(REPORT_AREA)


## 11. Один тип комнаты внутри района и вне его


In [ ]:
inside = df.loc[(df['room_type'] == top_room) & (df['neighbourhood'] == busy_area), 'price']
outside = df.loc[(df['room_type'] == top_room) & (df['neighbourhood'] != busy_area), 'price']
median_inside = float(inside.median())
median_outside = float(outside.median())
price_gap = median_inside - median_outside
side = 'дороже' if price_gap > 0 else 'дешевле' if price_gap < 0 else 'столько же'
REPORT_GAP = (
    f'Тип {top_room} в районе {busy_area} имеет медиану цены {median_inside:.1f}, '
    f'а тот же тип вне этого района — {median_outside:.1f}. '
    f'Внутри района такой тип {side}, разница {price_gap:.1f}.'
)
print(REPORT_GAP)


## 12. Текст для владельца


In [ ]:
REPORT_FINAL = (
    f'По всей таблице средняя цена {price_mean:.1f}, медиана {price_median:.1f}, '
    f'самая дешёвая ночь {price_min:.0f}, самая дорогая {price_max:.0f}. '
    f'Честной ориентировочной ценой беру медиану {price_median:.0f}: '
    f'среднее выше неё из-за дорогого хвоста, и на него нельзя равнять обычное объявление. '
    f'Самый наполненный район — {busy_area}: объявлений {n_busy}, медиана {median_busy:.1f}, '
    f'дороже общей медианы {n_busy_above} объявлений. '
    f'По медиане район совпадает с таблицей в целом, так что «много объявлений» здесь не значит «дороже». '
    f'Тип {top_room} внутри {busy_area} имеет медиану {median_inside:.1f}, '
    f'вне этого района — {median_outside:.1f}, разница {price_gap:.1f}. '
    f'Такой тип в самом наполненном районе дешевле, чем тот же тип в остальных районах. '
    f'Владельцу новой квартиры в {busy_area} ближе ориентир {median_inside:.0f} для типа {top_room}, '
    f'а не средняя цена по всей таблице. '
    f'Дальше этого отчёт не идёт: это одна таблица, близость числа гостей к цене не причина цены, '
    f'и ту же цифру нельзя обещать в другом городе.'
)
print(REPORT_FINAL)
